# Wave Equation with NumPy

Evolve the scalar wave equation directly in NumPy before asking NRPy to generate C.

Navigation: [Index](../index.ipynb) |
Previous: [Reference Metric](../1-intro/reference_metric.ipynb) |
Next: [Method of Lines and Runge-Kutta](method_of_lines_and_rk.ipynb)

## Learning Goals

- Evolve the scalar wave equation on the same 3D Cartesian grid used by the legacy `nrpytutorial` NumPy example.
- Fill ghost zones with the same quadratic extrapolation face-update algorithm shown in the boundary-condition GIF.
- Apply those boundary conditions at the same RK4 substeps used by the legacy example.
- Compare the same `24^3`, `48^3`, and `96^3` grid family used for the legacy convergence evidence.

## Words for This Notebook

- **Ghost zone:** extra grid cells outside the physical domain used by finite-difference stencils near boundaries.
- **Quadratic extrapolation:** a boundary rule that fills ghost zones from three interior points with coefficients `3, -3, 1`.
- **Face update:** one pass over a boundary face, edge, or corner region.
- **RK4:** a four-stage Runge-Kutta method for advancing one time step.
- **Convergence:** the pattern that the error decreases when the grid is refined.

## Scalar Wave System

The Cartesian scalar wave equation is

$$\partial_t^2 u = c^2 \nabla^2 u.$$

Introducing $v = \partial_t u$ gives the first-order system

$$\partial_t u = v,\qquad \partial_t v = c^2 \nabla^2 u.$$

This notebook uses the same hand-written NumPy algorithm as the legacy tutorial: a cell-centered Cartesian cube, two ghost zones on every side, a fourth-order finite-difference Laplacian, quadratic extrapolation outer boundaries, and RK4 time integration.

## Step 1: Use the Same Grid as the Legacy NumPy Example

The live run below uses exactly the same grid constants as the legacy notebook:

- `domain_size = 1.0`
- `freeparam_sigma = 3.0`
- `Nx = Ny = Nz = 24`
- `NGHOSTS = 2`
- cell-centered coordinates on `[-domain_size, domain_size]^3`

The larger `48^3` and `96^3` grids are included later through the same recorded convergence data used by the legacy tutorial, because running those pure-Python loops interactively is slow.

In [ ]:
import numpy as np

domain_size = 1.0
freeparam_c = 1.0
freeparam_sigma = 3.0
freeparam_u_offset = 1.0

Nx = Ny = Nz = 24
NGHOSTS = 2

xmin = ymin = zmin = -domain_size
xmax = ymax = zmax = +domain_size

dx = (xmax - xmin) / Nx
dy = (ymax - ymin) / Ny
dz = (zmax - zmin) / Nz

xx = np.zeros(Nx + 2 * NGHOSTS)
yy = np.zeros(Ny + 2 * NGHOSTS)
zz = np.zeros(Nz + 2 * NGHOSTS)

for i in range(Nx + 2 * NGHOSTS):
    xx[i] = xmin + (i - NGHOSTS + 0.5) * dx
for j in range(Ny + 2 * NGHOSTS):
    yy[j] = ymin + (j - NGHOSTS + 0.5) * dy
for k in range(Nz + 2 * NGHOSTS):
    zz[k] = zmin + (k - NGHOSTS + 0.5) * dz

dt = 0.5 * min(dx, dy, dz) / freeparam_c
t_final = 0.5 * domain_size

grid_size = (Nx + 2 * NGHOSTS) * (Ny + 2 * NGHOSTS) * (Nz + 2 * NGHOSTS)
print("grid:", f"{Nx}^3 interior points plus {NGHOSTS} ghost zones")
print("dx:", f"{dx:.6e}", "dt:", f"{dt:.6e}", "t_final:", f"{t_final:.6e}")

## Step 2: Define the Same Flattened Grid Storage

The legacy example stores 3D gridfunctions as one-dimensional arrays, using an explicit index function. That matches the memory layout style used later by generated C code.

In [ ]:
def IDX3D(i, j, k):
    return i + (Nx + 2 * NGHOSTS) * (j + (Ny + 2 * NGHOSTS) * k)


u = np.zeros(grid_size)
v = np.zeros(grid_size)

u_k1 = np.zeros(grid_size)
v_k1 = np.zeros(grid_size)
u_k2 = np.zeros(grid_size)
v_k2 = np.zeros(grid_size)
u_k3 = np.zeros(grid_size)
v_k3 = np.zeros(grid_size)
u_k4 = np.zeros(grid_size)
v_k4 = np.zeros(grid_size)
u_tmp = np.zeros(grid_size)
v_tmp = np.zeros(grid_size)

print("allocated gridfunction entries:", grid_size)

## Step 3: Set the Same Spherical-Gaussian Initial Data

The exact solution is the same ingoing-plus-outgoing spherical Gaussian used by the legacy NumPy tutorial. The small positive offset keeps the denominator in the relative-error diagnostic away from zero.

In [ ]:
def exact_solution_single_pt_u(time, x_i, y_j, z_k):
    r = np.sqrt(x_i * x_i + y_j * y_j + z_k * z_k)
    outgoing = (r - freeparam_c * time) * np.exp(
        -((r - freeparam_c * time) ** 2) / (2.0 * freeparam_sigma**2)
    ) / r
    ingoing = (r + freeparam_c * time) * np.exp(
        -((r + freeparam_c * time) ** 2) / (2.0 * freeparam_sigma**2)
    ) / r
    return outgoing + ingoing + freeparam_u_offset


def exact_solution_single_pt_v(time, x_i, y_j, z_k):
    r = np.sqrt(x_i * x_i + y_j * y_j + z_k * z_k)
    c = freeparam_c
    sigma = freeparam_sigma
    return (
        c * np.exp(-((c * time + r) ** 2) / (2.0 * sigma**2)) / r
        - c * np.exp(-((-c * time + r) ** 2) / (2.0 * sigma**2)) / r
        + c
        * (-c * time + r) ** 2
        * np.exp(-((-c * time + r) ** 2) / (2.0 * sigma**2))
        / (sigma**2 * r)
        - c
        * (c * time + r) ** 2
        * np.exp(-((c * time + r) ** 2) / (2.0 * sigma**2))
        / (sigma**2 * r)
    )


def exact_solution_all_points(time, u_gf, v_gf):
    for k in range(Nz + 2 * NGHOSTS):
        for j in range(Ny + 2 * NGHOSTS):
            for i in range(Nx + 2 * NGHOSTS):
                idx = IDX3D(i, j, k)
                u_gf[idx] = exact_solution_single_pt_u(time, xx[i], yy[j], zz[k])
                v_gf[idx] = exact_solution_single_pt_v(time, xx[i], yy[j], zz[k])


exact_solution_all_points(0.0, u, v)
print("initial center u:", f"{u[IDX3D((Nx + 2 * NGHOSTS) // 2, (Ny + 2 * NGHOSTS) // 2, (Nz + 2 * NGHOSTS) // 2)]:.12f}")

## Step 4: Evaluate the Same Fourth-Order Interior RHS

Only interior points are updated by the right-hand side. Ghost zones are filled separately by the boundary-condition algorithm.

In [ ]:
def eval_rhs_all_interior_points(u_gf, v_gf, u_rhs, v_rhs):
    for k in range(NGHOSTS, Nz + NGHOSTS):
        for j in range(NGHOSTS, Ny + NGHOSTS):
            for i in range(NGHOSTS, Nx + NGHOSTS):
                idx = IDX3D(i, j, k)
                u_rhs[idx] = v_gf[idx]
                v_rhs[idx] = freeparam_c**2 * (
                    (
                        -1.0 / 12.0 * (u_gf[IDX3D(i - 2, j, k)] + u_gf[IDX3D(i + 2, j, k)])
                        + 4.0 / 3.0 * (u_gf[IDX3D(i - 1, j, k)] + u_gf[IDX3D(i + 1, j, k)])
                        - 5.0 / 2.0 * u_gf[idx]
                    )
                    / (dx * dx)
                    + (
                        -1.0 / 12.0 * (u_gf[IDX3D(i, j - 2, k)] + u_gf[IDX3D(i, j + 2, k)])
                        + 4.0 / 3.0 * (u_gf[IDX3D(i, j - 1, k)] + u_gf[IDX3D(i, j + 1, k)])
                        - 5.0 / 2.0 * u_gf[idx]
                    )
                    / (dy * dy)
                    + (
                        -1.0 / 12.0 * (u_gf[IDX3D(i, j, k - 2)] + u_gf[IDX3D(i, j, k + 2)])
                        + 4.0 / 3.0 * (u_gf[IDX3D(i, j, k - 1)] + u_gf[IDX3D(i, j, k + 1)])
                        - 5.0 / 2.0 * u_gf[idx]
                    )
                    / (dz * dz)
                )


eval_rhs_all_interior_points(u, v, u_k1, v_k1)
print("sample RHS at center:", f"{v_k1[IDX3D((Nx + 2 * NGHOSTS) // 2, (Ny + 2 * NGHOSTS) // 2, (Nz + 2 * NGHOSTS) // 2)]:.12e}")

## Step 5: Apply the Same Boundary-Condition Update Algorithm

The GIF below is the one used by the legacy NumPy tutorial. The algorithm fills one ghost-zone layer at a time, from the innermost layer outward. After each face update, the active box expands so edges and corners can use already-filled neighboring ghost-zone data.

<img src="bdrycond_general_algorithm.gif" width="450" align="center">

For each face point, the quadratic extrapolation rule is

$$u_{\rm ghost} = 3u_1 - 3u_2 + u_3,$$

where $u_1$, $u_2$, and $u_3$ are the next three points in the inward face-normal direction. The code below matches the legacy `bc_face_update()` and `apply_extrapolation_bcs()` structure.

In [ ]:
def bc_face_update(gf, i0min, i0max, i1min, i1max, i2min, i2max, FACEX0, FACEX1, FACEX2):
    for i2 in range(i2min, i2max):
        for i1 in range(i1min, i1max):
            for i0 in range(i0min, i0max):
                gf[IDX3D(i0, i1, i2)] = (
                    +3.0 * gf[IDX3D(i0 + FACEX0, i1 + FACEX1, i2 + FACEX2)]
                    - 3.0 * gf[IDX3D(i0 + 2 * FACEX0, i1 + 2 * FACEX1, i2 + 2 * FACEX2)]
                    + 1.0 * gf[IDX3D(i0 + 3 * FACEX0, i1 + 3 * FACEX1, i2 + 3 * FACEX2)]
                )


MAXFACE = -1
NUL = 0
MINFACE = +1


def apply_extrapolation_bcs(u_gf, v_gf):
    for gf in [u_gf, v_gf]:
        imin = [NGHOSTS, NGHOSTS, NGHOSTS]
        imax = [Nx + NGHOSTS, Ny + NGHOSTS, Nz + NGHOSTS]
        for _which_gz in range(NGHOSTS):
            bc_face_update(gf, imin[0] - 1, imin[0], imin[1], imax[1], imin[2], imax[2], MINFACE, NUL, NUL)
            imin[0] -= 1
            bc_face_update(gf, imax[0], imax[0] + 1, imin[1], imax[1], imin[2], imax[2], MAXFACE, NUL, NUL)
            imax[0] += 1
            bc_face_update(gf, imin[0], imax[0], imin[1] - 1, imin[1], imin[2], imax[2], NUL, MINFACE, NUL)
            imin[1] -= 1
            bc_face_update(gf, imin[0], imax[0], imax[1], imax[1] + 1, imin[2], imax[2], NUL, MAXFACE, NUL)
            imax[1] += 1
            bc_face_update(gf, imin[0], imax[0], imin[1], imax[1], imin[2] - 1, imin[2], NUL, NUL, MINFACE)
            imin[2] -= 1
            bc_face_update(gf, imin[0], imax[0], imin[1], imax[1], imax[2], imax[2] + 1, NUL, NUL, MAXFACE)
            imax[2] += 1


apply_extrapolation_bcs(u, v)
print("lower-x ghost u sample:", f"{u[IDX3D(NGHOSTS - 1, NGHOSTS, NGHOSTS)]:.12f}")

## Step 6: Apply Boundary Conditions at the Same RK4 Substeps

The legacy algorithm applies boundary conditions to each temporary RK state before the next right-hand side is evaluated, then applies boundary conditions once more after the final update.

In [ ]:
def one_RK_step():
    eval_rhs_all_interior_points(u, v, u_k1, v_k1)

    for idx in range(grid_size):
        u_tmp[idx] = u[idx] + 0.5 * dt * u_k1[idx]
        v_tmp[idx] = v[idx] + 0.5 * dt * v_k1[idx]
    apply_extrapolation_bcs(u_tmp, v_tmp)

    eval_rhs_all_interior_points(u_tmp, v_tmp, u_k2, v_k2)

    for idx in range(grid_size):
        u_tmp[idx] = u[idx] + 0.5 * dt * u_k2[idx]
        v_tmp[idx] = v[idx] + 0.5 * dt * v_k2[idx]
    apply_extrapolation_bcs(u_tmp, v_tmp)

    eval_rhs_all_interior_points(u_tmp, v_tmp, u_k3, v_k3)

    for idx in range(grid_size):
        u_tmp[idx] = u[idx] + dt * u_k3[idx]
        v_tmp[idx] = v[idx] + dt * v_k3[idx]
    apply_extrapolation_bcs(u_tmp, v_tmp)

    eval_rhs_all_interior_points(u_tmp, v_tmp, u_k4, v_k4)

    for idx in range(grid_size):
        u[idx] = u[idx] + dt * (u_k1[idx] + 2.0 * u_k2[idx] + 2.0 * u_k3[idx] + u_k4[idx]) / 6.0
        v[idx] = v[idx] + dt * (v_k1[idx] + 2.0 * v_k2[idx] + 2.0 * v_k3[idx] + v_k4[idx]) / 6.0
    apply_extrapolation_bcs(u, v)


one_RK_step()
print("one-step center u:", f"{u[IDX3D((Nx + 2 * NGHOSTS) // 2, (Ny + 2 * NGHOSTS) // 2, (Nz + 2 * NGHOSTS) // 2)]:.12f}")

## Step 7: Run the Same `24^3` Live Example

This cell resets the gridfunctions to the exact initial data, evolves the same `24^3` grid used by the legacy notebook to `t = 0.5`, and records the same center-point diagnostic: time, `log10` relative error, numerical value, and exact value.

In [ ]:
exact_solution_all_points(0.0, u, v)

io = (Nx + 2 * NGHOSTS) // 2
jo = (Ny + 2 * NGHOSTS) // 2
ko = (Nz + 2 * NGHOSTS) // 2
print("diagnostic point:", f"({xx[io]:.2f}, {yy[jo]:.2f}, {zz[ko]:.2f})")


def diagnostics(n):
    curr_time = n * dt
    numerical = u[IDX3D(io, jo, ko)]
    exact = exact_solution_single_pt_u(curr_time, xx[io], yy[jo], zz[ko])
    log10relerror = np.log10(max(1.0e-16, abs((numerical - exact) / exact)))
    return curr_time, log10relerror, numerical, exact


n_final = int(t_final / dt + 0.5)
n_out_every = max(1, int(Nx / 24.0))
run_rows = [diagnostics(0)]

for n in range(n_final):
    one_RK_step()
    if (n + 1) % n_out_every == 0:
        run_rows.append(diagnostics(n + 1))

print("time log10_relative_error numerical exact")
for row in run_rows:
    print(f"{row[0]:.2f} {row[1]:.2f} {row[2]:.12f} {row[3]:.12f}")

## Step 8: Use the Same `24^3`, `48^3`, and `96^3` Grid Family for Convergence

The legacy tutorial records results for `24^3`, `48^3`, and `96^3`. Those are the same grids used here. Running `48^3` and `96^3` in pure Python is intentionally avoided in the live notebook, but the convergence check below uses the same recorded data pattern from that example.

In [ ]:
legacy_results = {
    24: '0.00 -16.00 2.999421380013 2.999421380013\n0.04 -10.70 2.998843001874 2.998843001814\n0.08 -10.06 2.997108425138 2.997108424880\n0.12 -9.70 2.994219322036 2.994219321440\n0.17 -9.45 2.990178477110 2.990178476038\n0.21 -9.25 2.984989783456 2.984989781772\n0.25 -9.09 2.978658237475 2.978658235046\n0.29 -8.95 2.971189932138 2.971189928832\n0.33 -8.84 2.962592048775 2.962592044465\n0.38 -8.73 2.952872847425 2.952872841973\n0.42 -8.64 2.942041655765 2.942041648971\n0.46 -8.54 2.930108856521 2.930108848127\n0.50 -8.48 2.917085872939 2.917085863230\n',
    48: '0.00 -16.00 2.999855329307 2.999855329307\n0.04 -11.87 2.999276741878 2.999276741874\n0.08 -11.25 2.997541537534 2.997541537518\n0.12 -10.89 2.994651389354 2.994651389316\n0.17 -10.64 2.990609083291 2.990609083222\n0.21 -10.45 2.985418514411 2.985418514305\n0.25 -10.29 2.979084681648 2.979084681494\n0.29 -10.15 2.971613681053 2.971613680844\n0.33 -10.04 2.963012697588 2.963012697316\n0.38 -9.93 2.953289995451 2.953289995108\n0.42 -9.84 2.942454906957 2.942454906535\n0.46 -9.76 2.930517820003 2.930517819496\n0.50 -9.69 2.917490164124 2.917490163524\n',
    96: '0.00 -16.00 2.999963831346 2.999963831346\n0.04 -13.06 2.999385191594 2.999385191594\n0.08 -12.45 2.997649830354 2.997649830353\n0.12 -12.09 2.994759420914 2.994759420911\n0.17 -11.84 2.990716749579 2.990716749575\n0.21 -11.65 2.985525711912 2.985525711906\n0.25 -11.49 2.979191307475 2.979191307466\n0.29 -11.36 2.971719633092 2.971719633079\n0.33 -11.24 2.963117874631 2.963117874614\n0.38 -11.14 2.953394297333 2.953394297311\n0.42 -11.05 2.942558234689 2.942558234663\n0.46 -10.97 2.930620075904 2.930620075872\n0.50 -10.89 2.917591251949 2.917591251911\n'
}

parsed_results = {}
for resolution, text in legacy_results.items():
    parsed_results[resolution] = np.loadtxt(text.splitlines())

print("resolution final_log10_relative_error")
for resolution in [24, 48, 96]:
    print(resolution, f"{parsed_results[resolution][-1, 1]:.2f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
for resolution, color in [(24, "tab:blue"), (48, "tab:orange"), (96, "tab:green")]:
    data = parsed_results[resolution]
    shifted_log_error = data[:, 1] + 4.0 * np.log10(resolution / 24.0)
    plt.plot(data[:, 0], shifted_log_error, color=color, label=f"{resolution}^3, shifted")

plt.xlabel("time")
plt.ylabel("shifted log10(relative error)")
plt.title("Fourth-order convergence on the same grid family")
plt.legend()
plt.show()

final_24 = parsed_results[24][-1, 1]
final_48_shifted = parsed_results[48][-1, 1] + 4.0 * np.log10(48.0 / 24.0)
final_96_shifted = parsed_results[96][-1, 1] + 4.0 * np.log10(96.0 / 24.0)
if abs(final_24 - final_48_shifted) > 0.15 or abs(final_24 - final_96_shifted) > 0.15:
    raise RuntimeError("Expected the shifted convergence curves to overlap.")

## What Changed from the Periodic Version

This notebook now uses the same boundary-condition update algorithm as the legacy NumPy scalar-wave tutorial. It no longer uses periodic wrapping. Ghost zones are explicit, and the extrapolation update happens at each temporary RK4 state before the next RHS evaluation.

The next notebook separates the Method-of-Lines idea from this concrete implementation.

## Learning Check

Point to the line in `apply_extrapolation_bcs()` where the active box expands after a face is filled. Explain why that matters for edges and corners.

## Continue to Method of Lines

- [Method of Lines and Runge-Kutta](method_of_lines_and_rk.ipynb)
- [Boundary Conditions and Convergence](boundary_conditions_and_convergence.ipynb)
- [Wave Equation and C Code Generation](../3-wave_equation/wave_equation_and_c_codegen.ipynb)